# 🏙️ NYC Employee Payroll Data Pipeline
### End-to-End Batch Data Engineering — Production

---

**Job:** `payroll_employees_nyc_prod` | **Job ID:** `37858787129397`

**Repository:** `github.com/Sh4444/databricks-data-engineering-projects` (branch `main`)

**Prepared by:** Sharful Haque · `sharfulhaque10@gmail.com`

**Date:** September 2026

---

> *Transforming raw NYC payroll CSV data into a fully audited, SCD Type 2–enabled, production-grade analytics pipeline on Databricks.*

# 📋 Agenda

| # | Topic | Duration |
|---|-------|----------|
| 1 | Business Context & Objectives | 5 min |
| 2 | Pipeline Architecture Overview | 5 min |
| 3 | Data Source & Schema Details | 5 min |
| 4 | Task-by-Task Deep Dive (6 tasks) | 20 min |
| 5 | Data Quality & Validation Framework | 5 min |
| 6 | Audit & Observability | 5 min |
| 7 | SCD Type 2 Implementation | 5 min |
| 8 | Schedule, Infrastructure & Governance | 5 min |
| 9 | Run History & Current Status | 5 min |
| 10 | Challenges, Lessons & Future Roadmap | 5 min |
| 11 | Q&A / Open Discussion | 10 min |

---

*Total estimated time: ~75 minutes*

# 🎯 Business Context & Objectives

## Background

The City of New York publishes payroll data for municipal employees — covering fiscal years, agencies, titles, salaries, overtime, and leave status. This dataset contains **500,000+ records per batch** and is critical for:

* **Transparency & accountability** in public sector compensation
* **Workforce analytics** — headcount, salary distributions, overtime trends
* **Budget planning** — forecasting payroll expenditure by agency and borough
* **Compliance reporting** — ensuring data accuracy and auditability

## Pipeline Objectives

| Objective | Description |
|---|---|
| **Ingest** | Read raw payroll CSV from UC Volume into Databricks |
| **Validate** | Apply schema enforcement, null checks, and business rules |
| **Route** | Separate clean records from rejected/quarantined records |
| **Track History** | Maintain SCD Type 2 for full change-data history |
| **Alert** | Notify stakeholders when rejections exceed thresholds |
| **Archive** | Move processed source files to timestamped archive |
| **Serve** | Refresh downstream SQL views for BI consumption |
| **Audit** | Log every step with metrics for full traceability |

## Key Stakeholders

* **Data Engineering** — Pipeline build, monitoring, and maintenance
* **Analytics / BI** — Downstream view consumers for dashboards & reporting
* **Compliance** — Audit trail and data lineage verification

# 🏗️ Pipeline Architecture Overview

## End-to-End Task Flow

```mermaid
flowchart TD
    CRON["Cron Trigger - Daily 6:56 PM IST - PAUSED"]
    REPO["GitHub Repo - branch: main"]
    
    FW["1. FW - Framework - notebooks/fwnb"]
    PVT["2. PVT - Pivot Transform - notebooks/01_pvtnb"]
    SCD["3. SCD - Slowly Changing Dim - notebooks/02_scdnb"]
    REJ["REJ_ALERTS - SQL Alert Task"]
    ARCHIVE["4. ARCHIVE - notebooks/03_archivenb"]
    VIEW["5. View_Refresh - views/views.sql"]
    
    CRON --> FW
    REPO -.->|source| FW
    REPO -.->|source| PVT
    REPO -.->|source| SCD
    REPO -.->|source| ARCHIVE
    REPO -.->|source| VIEW
    
    FW -->|ALL_SUCCESS| PVT
    PVT -->|ALL_SUCCESS| SCD
    PVT -->|ALL_SUCCESS| REJ
    SCD -->|ALL_SUCCESS| ARCHIVE
    SCD -->|ALL_SUCCESS| VIEW
```

## High-Level Data Flow

```
CSV (UC Volume) -> Framework -> Pivot (Ingest + Validate + Route)
                                      |
                         +-------------+--------------+
                         |             |              |
                    stg_table    quarantine     audit logs
                         |
                    SCD2 MERGE -> payroll_employee_nyc (Type 2)
                         |
                    +----+----+
                     |         |
                 Archive   View Refresh -> BI/Dashboards
```

## Design Principles

* **Medallion-lite architecture** — Raw -> Staging -> Curated (SCD2 target)
* **Idempotent batch processing** — Each run processes one CSV file end-to-end
* **Fail-fast with audit** — Errors are logged before raising exceptions
* **Git-backed** — All notebook & SQL source lives in GitHub, not inline

# 📥 Data Source & Schema

## Source File

| Property | Value |
|---|---|
| **Path** | `/Volumes/workspace/shprod/payrolldata/data/payrolldata.csv` |
| **Format** | CSV (header = true, inferSchema = false) |
| **Typical Volume** | ~509,525 records per batch |
| **Archive Path** | `/Volumes/workspace/shprod/payrolldata/archive/nycpayrolldata/` |

## Raw CSV Schema (16 Columns)

| # | Source Column | Transformed Column | Databricks Type |
|---|---|---|---|
| 1 | `Fiscal Year` | `fiscal_year` | `INT` |
| 2 | `Agency Name` | `agency_name` | `STRING` (UPPER TRIM) |
| 3 | `Last Name` | `last_name` | `STRING` (UPPER TRIM) |
| 4 | `First Name` | `first_name` | `STRING` (UPPER TRIM) |
| 5 | `Mid Init` | `mid_name` | `STRING` (UPPER TRIM) |
| 6 | `Agency Start Date` | `agency_start_date` | `DATE` (MM/dd/yyyy) |
| 7 | `Work Location Borough` | `work_location_borough` | `STRING` (UPPER TRIM) |
| 8 | `Title Description` | `title_description` | `STRING` (UPPER TRIM) |
| 9 | `Leave Status as of June 30` | `leave_status_as_of_june_30` | `STRING` (UPPER TRIM) |
| 10 | `Base Salary` | `base_salary` | `DECIMAL(12,2)` ($ , stripped) |
| 11 | `Pay Basis` | `pay_basis` | `STRING` (UPPER TRIM) |
| 12 | `Regular Hours` | `regular_hours` | `DECIMAL(10,2)` (, stripped) |
| 13 | `Regular Gross Paid` | `regular_gross_paid` | `DECIMAL(12,2)` ($ , stripped) |
| 14 | `OT Hours` | `ot_hours` | `DECIMAL(10,2)` (, stripped) |
| 15 | `Total OT Paid` | `total_ot_paid` | `DECIMAL(12,2)` ($ , stripped) |
| 16 | `Total Other Pay` | `total_other_pay` | `DECIMAL(12,2)` ($ , stripped) |

## Surrogate Key Generation

* **Key column:** `pid`
* **Algorithm:** `abs(xxhash64(concat_ws('||', first_name, last_name, agency_name, title_description, base_salary, base_salary, agency_start_date)))`
* **Purpose:** Deterministic deduplication — same employee record produces the same `pid` across runs
* **Rationale:** NYC payroll data has no natural unique key; composite hash provides a stable identifier for SCD2 MERGE

# 🔧 Task 1: FW — Framework Initialization

## Overview

| Property | Value |
|---|---|
| **Task Key** | `FW` |
| **Type** | Notebook (Git source) |
| **Notebook** | `notebooks/fwnb` |
| **Depends On** | None (entry point) |
| **Run If** | `ALL_SUCCESS` |

## Purpose

The Framework task is the **entry point** of the pipeline. It performs initial setup that downstream tasks depend on:

* **Environment initialization** — Spark session, imports, common utilities
* **Audit utility loading** — Loads the shared `PipelineLogger` class via `%run ./00_audit_utilsnb.ipynb`
* **Run ID allocation** — Queries `shprod.audit_runlog` for the next sequential `run_id`
* **Task value propagation** — Sets `pipeline_run_id` via `dbutils.jobs.taskValues.set()` for downstream consumption

## Key Code Snippet

```python
get_run_id = spark.sql(
    "SELECT COALESCE(MAX(run_id), 0) + 1 AS next_id FROM shprod.audit_runlog"
).collect()[0]["next_id"]

logger = PipelineLogger(
    spark=spark,
    pipeline_name="NYC_PAYROLL_BATCH_PVT_PIPELINE",
    run_id=get_run_id
)
logger.open_runlog()

dbutils.jobs.taskValues.set(key="pipeline_run_id", value=logger.run_id)
```

## Why It Matters

* Ensures **every pipeline run has a unique, traceable run_id**
* The `open_runlog()` inserts a `RUNNING` status row — enabling real-time monitoring
* Downstream tasks retrieve this `run_id` to maintain audit continuity across the entire DAG

# 🔄 Task 2: PVT — Ingestion, Validation & Routing

## Overview

| Property | Value |
|---|---|
| **Task Key** | `PVT` |
| **Type** | Notebook (Git source) |
| **Notebook** | `notebooks/01_pvtnb` |
| **Depends On** | `FW` |
| **Run If** | `ALL_SUCCESS` |

## Purpose

This is the **core transformation task** — it reads the raw CSV, standardizes columns, generates surrogate keys, validates records, and routes data to staging or quarantine.

## Processing Steps

### Step 1: Read Raw CSV
```python
raw_df = spark.read.option("header", "true").option("inferSchema", "false").csv(source_file_path)
rows_read = raw_df.count()  # ~509,525 rows
```

### Step 2: Standardize & Generate Surrogate Key
* 16 raw columns mapped to typed Spark columns
* `UPPER()` + `TRIM()` applied to all string fields
* Currency columns (`$`, `,`) stripped and cast to `DECIMAL`
* Dates parsed from `MM/dd/yyyy` format
* `pid` generated via `xxhash64()` of composite key
* `run_id` attached as audit column

### Step 3: Validation Rules
```python
valid_cond = (
    col("pid").isNotNull() &
    col("fiscal_year").isNotNull() &
    (col("first_name").isNotNull() | col("last_name").isNotNull()) &
    col("title_description").isNotNull() &
    (col("base_salary") >= 0)
)
```

### Step 4: Route Records

| Destination | Condition | Action |
|---|---|---|
| **Valid** | Passes all validation rules | Deduplicate on `pid`, write to `shprod.stg_payroll_employee_nyc` |
| **Invalid** | Fails validation | Write to `shprod.payroll_quarantine` with `rejection_reason` + `quarantined_at` |
| **Duplicates** | Same `pid` already seen | Write to quarantine with reason `Duplicate Identifier - Dropped` |
| **History** | Copy of staging table | Append to `shprod.hist_payroll_employee_nyc` for point-in-time recovery |

## Rejection Reason Taxonomy

* `NULL or Invalid Fiscal Year`
* `Negative Base Salary`
* `NULL Title Description`
* `Missing Name or PID`
* `Duplicate Identifier - Dropped`

## Audit Logging

All steps are logged via:
```python
logger.log_step(
    step_name="01_PVT_INGEST",
    source_table=source_file_path,
    target_table="shprod.stg_payroll_employee_nyc",
    error_table="shprod.payroll_quarantine",
    status="SUCCESS",
    rows_read=rows_read,
    rows_inserted=rows_inserted,
    rows_quarantined=rows_quarantined,
    ...
)
```

# 📊 Task 3: SCD — Slowly Changing Dimensions Type 2

## Overview

| Property | Value |
|---|---|
| **Task Key** | `SCD` |
| **Type** | Notebook (Git source) |
| **Notebook** | `notebooks/02_scdnb` |
| **Depends On** | `PVT` |
| **Run If** | `ALL_SUCCESS` |

## Purpose

Implements **SCD Type 2** on the curated target table `shprod.payroll_employee_nyc` — tracking the full history of changes to employee records over time.

## Source & Target

| Role | Table |
|---|---|
| **Source** | `shprod.stg_payroll_employee_nyc` (staging — current batch) |
| **Target** | `shprod.payroll_employee_nyc` (SCD2 curated table) |

## SCD Type 2 Strategy — Dual-Leg MERGE

### Leg 1: Changed Active Records (Close Out)
* Join staging with **currently active** target rows (`end_dt = '9999-12-31'`)
* Compare 11 tracked columns for changes:
  * `agency_name`, `title_description`, `work_location_borough`, `leave_status_as_of_june_30`
  * `base_salary`, `pay_basis`, `regular_hours`, `regular_gross_paid`
  * `ot_hours`, `total_ot_paid`, `total_other_pay`
* If any column changed -> set `end_dt` to the new `active_dt` (retire the old version)

### Leg 2: New + Changed Records (Insert)
* All staging rows (new + changed versions) get `merge_key = NULL`
* Inserted with `end_dt = '9999-12-31'` (sentinel for active)

## SCD2 Column Design

| Column | Type | Purpose |
|---|---|---|
| `active_dt` | `DATE` | When this version became active |
| `end_dt` | `DATE` | When this version was retired (`9999-12-31` = still active) |
| `run_id` | `INT` | Run that first inserted this record |
| `lst_uptd_run_id` | `INT` | Run that last updated (retired) this record |

## Metrics Capture

After MERGE, Delta operation metrics are extracted:
```python
last_op_metrics = target_table.history(1).select("operationMetrics").collect()[0][0]
num_inserted = int(last_op_metrics.get("numTargetRowsInserted", 0))
num_updated = int(last_op_metrics.get("numTargetRowsUpdated", 0))
```

These metrics are logged to `shprod.audit_execution_logs` for full traceability.

# 🔔 Task 4: REJ_ALERTS — Rejection Alerting

## Overview

| Property | Value |
|---|---|
| **Task Key** | `REJ_ALERTS` |
| **Type** | SQL Alert Task |
| **Alert ID** | `1208922090064189` |
| **Warehouse** | `dc31fdf744455703` (SQL Warehouse) |
| **Depends On** | `PVT` |
| **Run If** | `ALL_SUCCESS` |
| **Subscriber** | Configured destination ID `35f202d3-1ddb-4ef5-95ad-4a9294afea8f` |

## Purpose

After the PVT task quarantines invalid/duplicate records, this SQL Alert evaluates whether the rejection count crosses a defined threshold. If triggered:

* **Stakeholders are automatically notified** via the alert subscriber
* **Action can be taken** to investigate data quality issues before they propagate

## How It Works

1. The alert runs a SQL query against `shprod.payroll_quarantine` (or a related metric query)
2. If the result exceeds the configured threshold -> alert fires
3. Notification is sent to the configured subscriber destination
4. The alert runs on the **SQL Warehouse** (`dc31fdf744455703`) — no Spark cluster needed

## Position in the DAG

```
PVT (Pivot) --+-- SCD --+-- ARCHIVE
              |         +-- View_Refresh
              +-- REJ_ALERTS  <- Runs in parallel with SCD
```

* `REJ_ALERTS` depends only on `PVT`, not `SCD` — it can run **in parallel** with the SCD task
* This reduces critical path latency — alerting doesn't block the main data flow

# 📦 Task 5: ARCHIVE — File Archiving

## Overview

| Property | Value |
|---|---|
| **Task Key** | `ARCHIVE` |
| **Type** | Notebook (Git source) |
| **Notebook** | `notebooks/03_archivenb` |
| **Depends On** | `SCD` |
| **Run If** | `ALL_SUCCESS` |

## Purpose

After the source CSV has been fully processed (ingested -> validated -> SCD2 merged), the original file is **moved to an archive location** with a timestamp prefix. This ensures:

* **No reprocessing** — The source directory is cleared for the next batch
* **Audit trail** — Every processed file is preserved with a timestamp
* **Reproducibility** — Archived files can be retrieved for debugging or re-runs

## Implementation

```python
def archive_file(source_path: str, archive_dir: str) -> str:
    dbutils.fs.ls(source_path)  # Verify source exists
    
    base_name = os.path.basename(source_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    target_path = f"{archive_dir}/{timestamp}_{base_name}"
    
    dbutils.fs.mv(source_path, target_path)  # Atomic move
    return target_path
```

## Paths

| Role | Path |
|---|---|
| **Source** | `/Volumes/workspace/shprod/payrolldata/data/payrolldata.csv` |
| **Archive** | `/Volumes/workspace/shprod/payrolldata/archive/nycpayrolldata/` |
| **Example Output** | `.../20260920_181745_payrolldata.csv` |

## Why It Runs Last (After SCD)

* The file must remain available until **all downstream tasks** (SCD, View_Refresh) have completed
* If archiving happened earlier and a downstream task needed to re-read the source, the file would be missing
* Placing ARCHIVE after SCD guarantees the file is only moved once processing is fully committed

# 🖥️ Task 6: View_Refresh — SQL Views Refresh

## Overview

| Property | Value |
|---|---|
| **Task Key** | `View_Refresh` |
| **Type** | SQL File Task (Git source) |
| **File** | `views/views.sql` |
| **Warehouse** | `dc31fdf744455703` (SQL Warehouse) |
| **Depends On** | `SCD` |
| **Run If** | `ALL_SUCCESS` |

## Purpose

After the SCD2 MERGE updates the curated table, this task **refreshes downstream SQL views** that BI dashboards and reporting tools depend on. These views typically:

* Expose the **currently active** employee records (`WHERE end_dt = '9999-12-31'`)
* Provide **aggregated summaries** (e.g., headcount by agency, average salary by borough)
* Join payroll data with **reference/dimension tables** for enrichment
* Create **consumer-friendly column names** and formats

## Execution Model

* Runs on the **SQL Warehouse** — no Spark cluster required
* Executes the SQL file `views/views.sql` from the Git repo
* Runs **in parallel** with ARCHIVE (both depend only on SCD)

## Why It Depends on SCD (Not PVT)

* Views read from the **curated SCD2 table** (`shprod.payroll_employee_nyc`), not the staging table
* The SCD task must complete the MERGE before views can reflect the latest data
* This ensures **dashboards always show the most current, validated, and history-tracked data**

# ✅ Data Quality & Validation Framework

## Multi-Layer Validation Strategy

The pipeline implements validation at **three distinct layers**:

### Layer 1: Schema Enforcement (Ingestion)
* All 16 CSV columns are explicitly cast to typed Spark columns
* String columns: `UPPER(TRIM())` normalization
* Numeric columns: `$` and `,` stripped before `DECIMAL` cast
* Date columns: Parsed from `MM/dd/yyyy` format
* **No `inferSchema`** — all types are explicitly controlled

### Layer 2: Business Rule Validation (PVT)

| Rule | Condition | Failure Action |
|---|---|---|
| Non-null PID | `pid IS NOT NULL` | Quarantine: `Missing Name or PID` |
| Valid Fiscal Year | `fiscal_year IS NOT NULL` | Quarantine: `NULL or Invalid Fiscal Year` |
| Name Present | `first_name IS NOT NULL OR last_name IS NOT NULL` | Quarantine: `Missing Name or PID` |
| Title Present | `title_description IS NOT NULL` | Quarantine: `NULL Title Description` |
| Non-negative Salary | `base_salary >= 0` | Quarantine: `Negative Base Salary` |

### Layer 3: Deduplication (PVT)
* Records are deduplicated on `pid` using a **window function**:
  * `row_number() OVER (PARTITION BY pid ORDER BY agency_start_date ASC NULLS LAST)`
  * First occurrence (earliest start date) is kept
  * Subsequent duplicates are quarantined with reason `Duplicate Identifier - Dropped`

### Layer 4: Rejection Alerting (REJ_ALERTS)
* SQL Alert monitors quarantine table for threshold breaches
* Automated email notification to stakeholders

## Quarantine Table Schema

| Column | Description |
|---|---|
| All 16 standardized columns | Original data preserved for debugging |
| `rejection_reason` | Categorized reason for rejection |
| `quarantined_at` | Timestamp of quarantine |
| `run_id` | Links to the pipeline run that quarantined it |

## Data Quality Metrics Tracked

* `rows_read` — Total records in source CSV
* `rows_inserted` — Valid records written to staging
* `rows_quarantined` — Invalid/duplicate records rejected
* Quarantine rate = `rows_quarantined / rows_read` (monitored via alert)

# 🔍 Audit & Observability Framework

## Architecture

The pipeline features a **custom audit framework** built on two Unity Catalog tables, powered by a reusable `PipelineLogger` class.

## Audit Tables

### 1. `shprod.audit_runlog` — Run-Level Audit

| Column | Type | Description |
|---|---|---|
| `run_id` | INT | Unique, auto-incremented per pipeline run |
| `run_date` | DATE | Date of the run |
| `pipeline_name` | STRING | E.g., `NYC_PAYROLL_BATCH_PVT_PIPELINE` |
| `status` | STRING | `RUNNING` -> `COMPLETED` or `FAILED` |
| `start_time` | TIMESTAMP | When the run started |
| `end_time` | TIMESTAMP | When the run ended |
| `total_duration_sec` | INT | End-to-end duration in seconds |
| `run_by` | STRING | `current_user()` |

### 2. `shprod.audit_execution_logs` — Step-Level Audit

| Column | Type | Description |
|---|---|---|
| `log_id` | UUID | Unique per step execution |
| `run_id` | INT | FK to `audit_runlog` |
| `log_date` | DATE | Date of the step |
| `step_name` | STRING | E.g., `01_PVT_INGEST`, `02_SCD2_MERGE` |
| `source_table` | STRING | Input table/path |
| `target_table` | STRING | Output table |
| `error_table` | STRING | Quarantine table |
| `status` | STRING | `SUCCESS` or `FAILED` |
| `rows_read` | INT | Records processed |
| `rows_inserted` | INT | Records written |
| `rows_updated` | INT | Records updated (SCD2) |
| `rows_quarantined` | INT | Records rejected |
| `start_time` | TIMESTAMP | Step start |
| `end_time` | TIMESTAMP | Step end |
| `error_msg` | STRING | Truncated traceback if failed |

## PipelineLogger Class

```python
class PipelineLogger:
    def open_runlog(self):        # INSERT RUNNING row
    def close_runlog(self):       # UPDATE to COMPLETED/FAILED
    def log_step(self, ...):      # INSERT step execution record
    def update_active_dt(self):  # Set active date on staging
```

## Observability Benefits

* **Run-level traceability** — Every pipeline run has a unique ID with start/end/duration
* **Step-level granularity** — Know exactly which step failed and why
* **Row-count lineage** — Track records read, inserted, updated, quarantined at each step
* **Error capture** — Full traceback stored (truncated to 1000 chars) for debugging
* **Cross-task continuity** — `run_id` propagated via `dbutils.jobs.taskValues`

# 📈 SCD Type 2 Implementation Deep Dive

## What is SCD Type 2?

Slowly Changing Dimension Type 2 preserves **full historical context** by creating a new row for each change rather than overwriting the existing record. This enables point-in-time analysis: *What was this employee's salary in Q1 2024?*

## Table Structure Example

```
+-----+----------+---------------+------------+------------+--------+--------------+
| pid | salary   | title         | active_dt  | end_dt     | run_id | lst_uptd_run |
+-----+----------+---------------+------------+------------+--------+--------------+
| 101 | $65,000  | ANALYST       | 2024-01-01 | 2024-06-15 | 1      | 2            |  <- Retired
| 101 | $72,000  | SR. ANALYST   | 2024-06-15 | 9999-12-31 | 2      | 2            |  <- Active
+-----+----------+---------------+------------+------------+--------+--------------+
```

## MERGE Strategy — The Dual-Leg Pattern

```
Staging Table (current batch) -> Two-legged approach -> Single Delta MERGE

Leg 1: Find CHANGED active records -> merge_key = pid      -> UPDATE (retire old row)
Leg 2: All staging rows            -> merge_key = NULL     -> INSERT (new active row)
```

### Why Two Legs?

A single MERGE cannot simultaneously UPDATE and INSERT the same key. The dual-leg pattern:
1. **Leg 1** matches existing active rows and retires them (`end_dt = source.active_dt`)
2. **Leg 2** (with `merge_key = NULL`) never matches -> always inserts as a new active row

### Tracked Columns (11 attributes)

| Category | Columns |
|---|---|
| **Identity** | `agency_name`, `title_description` |
| **Location** | `work_location_borough` |
| **Status** | `leave_status_as_of_june_30` |
| **Compensation** | `base_salary`, `pay_basis` |
| **Hours & Pay** | `regular_hours`, `regular_gross_paid`, `ot_hours`, `total_ot_paid`, `total_other_pay` |

### Sentinel Date Pattern

* `end_dt = '9999-12-31'` -> record is **currently active**
* Any other `end_dt` -> record was **retired** on that date
* Querying active records: `WHERE end_dt = '9999-12-31'`
* Querying history: `WHERE pid = X ORDER BY active_dt`

## Atomicity Guarantee

* The MERGE is a **single atomic Delta operation** — no partial states
* If the MERGE fails, the entire operation rolls back
* Delta's `history()` API provides post-MERGE metrics for audit logging

## Performance Considerations

* Delta MERGE leverages **data skipping** on `pid` and `end_dt`
* Only active records (`end_dt = '9999-12-31'`) are joined — not the full history
* `eqNullSafe` used for null-safe comparisons on all tracked columns

# ⚙️ Schedule, Infrastructure & Governance

## Schedule Configuration

| Property | Value |
|---|---|
| **Cron Expression** | `24 56 18 * * ?` (daily at ~6:56 PM) |
| **Timezone** | `Asia/Calcutta` (IST) |
| **Pause Status** | `PAUSED` — currently manually triggered |
| **Max Concurrent Runs** | `1` — prevents overlapping batch runs |

## Infrastructure

| Component | Configuration |
|---|---|
| **Compute** | Serverless / Job cluster (auto-provisioned) |
| **SQL Warehouse** | `dc31fdf744455703` — used by REJ_ALERTS & View_Refresh |
| **Queue** | Enabled — runs wait in queue if cluster is busy |
| **Performance Target** | `PERFORMANCE_OPTIMIZED` |
| **Timeout** | No timeout (0 = unlimited) |

## Git Integration

| Property | Value |
|---|---|
| **Repository** | `github.com/Sh4444/databricks-data-engineering-projects` |
| **Provider** | GitHub |
| **Branch** | `main` |
| **Source for** | All notebook tasks + SQL file task |

## Notifications

| Event | Recipient |
|---|---|
| **On Success** | `sharfulhaque10@gmail.com` |
| **On Failure** | `sharfulhaque10@gmail.com` |
| **Skipped Runs** | Alert enabled (`no_alert_for_skipped_runs = false`) |

## Governance & Tags

| Tag Key | Tag Value |
|---|---|
| `PROD` | `Pipeline` |

* Tag enables easy filtering, cost tracking, and access control policies
* All tables reside in catalog `shprod` (Unity Catalog governed)

## Run-As Configuration

| Property | Value |
|---|---|
| **Run As** | Owner (`sharfulhaque10@gmail.com`) |
| **Mode** | `run_as_owner = true` |

* Ensures consistent permissions across all task executions
* Simplifies access management — the owner's UC permissions apply to all tasks

# 📊 Run History & Current Status

## Recent Pipeline Runs

| Run ID | Status | Trigger | Notes |
|---|---|---|---|
| `1065167128496499` | ✅ SUCCESS | One-time | Latest successful run |
| `799552613960445` | ❌ FAILED | One-time | `View_Refresh` failed — file `views/views` not found in Git repo |
| `502601233709049` | ✅ SUCCESS | One-time | Earlier successful run |

## Known Issue

* **Run `799552613960445`** failed because the Git path `views/views` was not found (missing `.sql` extension or file path mismatch)
* This was resolved in subsequent runs — the latest run completed successfully

## Schedule Status

* The cron schedule is currently **PAUSED**
* All recent runs were **manually triggered** (`ONE_TIME`)
* The schedule can be resumed when the pipeline is ready for production automation

## Recommendations

1. **Resume schedule** once confident in automated runs
2. **Add retry policy** for transient failures (e.g., warehouse startup)
3. **Monitor quarantine rate** over time to detect upstream data quality drift
4. **Set up SLA monitoring** — alert if pipeline duration exceeds expected threshold

# 💡 Challenges & Lessons Learned

## Challenges Encountered

### 1. No Natural Primary Key
* NYC payroll data has **no unique identifier** per employee record
* **Solution:** Composite `xxhash64()` surrogate key (`pid`) from 7 columns
* **Trade-off:** Hash collisions are possible but extremely rare with 64-bit hashing

### 2. SCD2 MERGE Complexity
* Standard MERGE cannot handle simultaneous update + insert of the same key
* **Solution:** Dual-leg pattern with `merge_key` trick (Leg 1: key = pid, Leg 2: key = NULL)
* **Lesson:** Delta MERGE with the dual-leg pattern is the industry-standard SCD2 approach

### 3. Git Source Path Resolution
* Run `799552613960445` failed because `views/views` was not found
* **Root Cause:** File path mismatch in Git repo structure
* **Solution:** Ensure Git paths match exactly (including `.sql` extension where needed)

### 4. Data Quality Variability
* Raw CSV contains nulls, duplicates, negative salaries, and formatting inconsistencies
* **Solution:** Multi-layer validation + quarantine table with categorized rejection reasons
* **Lesson:** Never trust raw data — validate at ingestion, route failures transparently

### 5. Currency & Date Parsing
* Salary fields contain `$`, `,` characters; dates in `MM/dd/yyyy` format
* **Solution:** `regexp_replace()` to strip currency symbols before casting
* **Lesson:** Explicit type casting (no `inferSchema`) prevents silent data corruption

## Key Lessons

* **Audit everything** — The `PipelineLogger` class provides full run + step traceability
* **Quarantine, don't discard** — Rejected records are preserved for investigation and reprocessing
* **Idempotent design** — Each run processes one file end-to-end; archived files prevent reprocessing
* **Git-backed source** — All code in GitHub enables version control, code review, and CI/CD

# 🚀 Future Enhancements & Roadmap

## Short-Term (Next 1-2 Sprints)

| # | Enhancement | Impact |
|---|---|---|
| 1 | **Resume cron schedule** — Unpause daily 6:56 PM IST trigger | Automated daily processing |
| 2 | **Add retry policy** — Retry transient failures (warehouse startup, network) | Improved reliability |
| 3 | **Fix Git path** — Ensure `views/views.sql` path is stable | Prevent `INVALID_RUN_CONFIGURATION` failures |
| 4 | **Add SLA alert** — Alert if pipeline duration > 30 min | Operational monitoring |
| 5 | **Delta table optimization** — `OPTIMIZE` + `ZORDER BY (pid, end_dt)` | Faster MERGE & queries |

## Medium-Term (Next Quarter)

| # | Enhancement | Impact |
|---|---|---|
| 1 | **Auto Loader** for streaming ingestion — Replace batch CSV read with file arrival trigger | Near real-time processing |
| 2 | **Data quality dashboard** — Visualize quarantine rates, rejection reasons over time | Proactive DQ monitoring |
| 3 | **Schema registry** — Centralize schema definitions instead of inline casting | Maintainability |
| 4 | **Unit tests** — Pytest suite for validation rules and SCD2 MERGE logic | Regression prevention |
| 5 | **CI/CD pipeline** — GitHub Actions to validate notebooks before merge | Code quality gate |

## Long-Term (Next 6 Months)

| # | Enhancement | Impact |
|---|---|---|
| 1 | **ML-based anomaly detection** — Detect unusual salary changes or OT patterns | Fraud detection |
| 2 | **Cross-agency analytics** — Join with budget, headcount, and performance datasets | Richer insights |
| 3 | **Self-service BI** — Genie space or dashboard for stakeholders to query payroll data | Democratized access |
| 4 | **Data contracts** — Define upstream SLAs with data providers | Upstream accountability |
| 5 | **Multi-region replication** — Delta sharing for disaster recovery | Business continuity |

## Technical Debt to Address

* Move `inferSchema = false` to a **schema registry** (currently inline)
* Add **column-level comments** to all Unity Catalog tables
* Implement **data retention policies** for quarantine and audit tables
* Consider **Delta Liquid Clustering** instead of `ZORDER BY` for adaptive optimization

# 🎯 Summary & Q&A

## Pipeline at a Glance

```
CSV (509K+ rows) -> Framework -> Pivot (Validate + Route) -> SCD2 MERGE -> Archive + Views
                          |                              |
                    Quarantine Table              Audit Logs (Run + Step)
                          |
                    Rejection Alert (Email)
```

## Key Achievements

* ✅ **End-to-end automation** — From raw CSV to curated, SCD2-tracked, view-served data
* ✅ **Full audit trail** — Every run and step logged with metrics and error traces
* ✅ **Data quality framework** — Multi-layer validation with categorized quarantine
* ✅ **SCD Type 2** — Complete historical change tracking for 11 attributes
* ✅ **Git-backed** — All source code version-controlled in GitHub
* ✅ **Production-ready** — Unity Catalog governed, tagged, and notification-enabled

## Architecture Strengths

| Strength | How |
|---|---|
| **Reliability** | Fail-fast with audit, atomic MERGE, email alerts |
| **Traceability** | Run-level + step-level audit with row counts |
| **Maintainability** | Git source, shared `PipelineLogger` utility |
| **Scalability** | Serverless compute, Delta MERGE, queue-enabled |
| **Data Quality** | Multi-layer validation, quarantine with reasons |

## Open Discussion

* What additional validation rules should we add?
* Should we move to Auto Loader for streaming ingestion?
* How do we handle schema evolution in the source CSV?
* Who should be additional alert subscribers?
* What BI dashboards should consume the refreshed views?

---

### Thank You!

**Contact:** Sharful Haque · `sharfulhaque10@gmail.com`

**Job:** [payroll_employees_nyc_prod](#job-37858787129397)

**Repo:** `github.com/Sh4444/databricks-data-engineering-projects`

---

*Questions? Discussion?*

# 📎 Appendix: Complete Task Reference

## All Tasks — Detailed Configuration

| # | Task Key | Type | Source | Depends On | Run If | Warehouse |
|---|---|---|---|---|---|---|
| 1 | `FW` | Notebook | `notebooks/fwnb` (Git) | — | ALL_SUCCESS | — |
| 2 | `PVT` | Notebook | `notebooks/01_pvtnb` (Git) | FW | ALL_SUCCESS | — |
| 3 | `REJ_ALERTS` | SQL Alert | alert_id `1208922090064189` | PVT | ALL_SUCCESS | `dc31fdf744455703` |
| 4 | `SCD` | Notebook | `notebooks/02_scdnb` (Git) | PVT | ALL_SUCCESS | — |
| 5 | `ARCHIVE` | Notebook | `notebooks/03_archivenb` (Git) | SCD | ALL_SUCCESS | — |
| 6 | `View_Refresh` | SQL File | `views/views.sql` (Git) | SCD | ALL_SUCCESS | `dc31fdf744455703` |

## Unity Catalog Tables

| Table | Purpose | Layer |
|---|---|---|
| `shprod.stg_payroll_employee_nyc` | Staging — current batch validated records | Silver |
| `shprod.payroll_employee_nyc` | Curated SCD2 target — full history | Gold |
| `shprod.hist_payroll_employee_nyc` | Point-in-time copy of staging | Bronze/Silver backup |
| `shprod.payroll_quarantine` | Rejected/invalid records with reasons | Quarantine |
| `shprod.audit_runlog` | Run-level audit log | Audit |
| `shprod.audit_execution_logs` | Step-level audit log with metrics | Audit |

## UC Volumes

| Volume Path | Purpose |
|---|---|
| `/Volumes/workspace/shprod/payrolldata/data/` | Active source file location |
| `/Volumes/workspace/shprod/payrolldata/archive/nycpayrolldata/` | Archived processed files |

## Shared Utility Notebook

| Notebook | Path | Purpose |
|---|---|---|
| `00_audit_utilsnb` | `notebooks/00_audit_utilsnb` | `PipelineLogger` class — run + step audit logging |

## Dependencies Graph (Adjacency)

```
FW      -> [PVT]
PVT     -> [SCD, REJ_ALERTS]
SCD     -> [ARCHIVE, View_Refresh]
REJ_ALERTS -> []  (terminal)
ARCHIVE  -> []  (terminal)
View_Refresh -> []  (terminal)
```

# 📎 Appendix: Key Code Patterns

## 1. Surrogate Key Generation (PVT)

```python
abs(xxhash64(
    concat_ws("||",
        coalesce(upper(trim(col("First Name"))), lit("")),
        coalesce(upper(trim(col("Last Name"))), lit("")),
        coalesce(upper(trim(col("Agency Name"))), lit("")),
        coalesce(upper(trim(col("Title Description"))), lit("")),
        coalesce(upper(trim(col("Base Salary"))), lit("")),
        coalesce(upper(trim(col("Base Salary"))), lit("")),
        coalesce(to_date(col("Agency Start Date"), "MM/dd/yyyy").cast("string"), lit(""))
    )
)).alias("pid")
```

## 2. Validation + Rejection Routing (PVT)

```python
valid_cond = (
    col("pid").isNotNull() &
    col("fiscal_year").isNotNull() &
    (col("first_name").isNotNull() | col("last_name").isNotNull()) &
    col("title_description").isNotNull() &
    (col("base_salary") >= 0)
)

invalid_df = standardized_df.filter(~valid_cond)
    .withColumn("rejection_reason",
        when(col("fiscal_year").isNull(), lit("NULL or Invalid Fiscal Year"))
        .when(col("base_salary") < 0, lit("Negative Base Salary"))
        .when(col("title_description").isNull(), lit("NULL Title Description"))
        .otherwise(lit("Missing Name or PID"))
    )
    .withColumn("quarantined_at", current_timestamp())
```

## 3. Deduplication via Window Function (PVT)

```python
dedup_window = Window.partitionBy("pid").orderBy(col("agency_start_date").asc_nulls_last())
ranked_valid_df = valid_df.withColumn("rn", row_number().over(dedup_window))
clean_df = ranked_valid_df.filter(col("rn") == 1).drop("rn")
```

## 4. SCD2 Dual-Leg MERGE (SCD)

```python
# Leg 1: Changed records (retire old version)
changed_records_df = (
    stg_df.alias("s")
    .join(target_active_df.alias("t"), col("s.pid") == col("t.pid"), "inner")
    .filter(/* column change conditions */)
    .select("s.*")
    .withColumn("merge_key", col("s.pid"))
)

# Leg 2: All staging rows (new + changed -> insert)
insert_records_df = stg_df.withColumn("merge_key", lit(None).cast("string"))

# Union and MERGE
staged_updates_df = changed_records_df.unionByName(insert_records_df)

target_table.alias("target").merge(
    staged_updates_df.alias("source"),
    "target.pid = source.merge_key AND target.end_dt = CAST('9999-12-31' AS DATE)"
).whenMatchedUpdate(
    set={"end_dt": to_date(col("source.active_dt")), "lst_uptd_run_id": col("source.run_id")}
).whenNotMatchedInsert(
    values={...all columns..., "end_dt": to_date(lit("9999-12-31")), ...}
).execute()
```

## 5. File Archival (ARCHIVE)

```python
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
target_path = f"{archive_dir}/{timestamp}_{base_name}"
dbutils.fs.mv(source_path, target_path)
```

## 6. Audit Logging Pattern (All Tasks)

```python
try:
    # ... processing logic ...
    logger.log_step(
        step_name="STEP_NAME",
        source_table=source, target_table=target,
        error_table="payroll_quarantine",
        status="SUCCESS",
        rows_read=N, rows_inserted=N, rows_updated=N, rows_quarantined=N,
        start_time=step_start, end_time=datetime.now()
    )
    logger.close_runlog()
except Exception as e:
    err_msg = str(traceback.format_exc()).replace("'", "\"")[:1000]
    logger.log_step(..., status="FAILED", error_msg=err_msg)
    logger.close_runlog(status="FAILED")
    raise e
```